# TalentSync AI - Fine-Tuned Model Evaluation

This notebook evaluates the performance of the newly fine-tuned embedding model.

In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load fine-tuned model
model = SentenceTransformer('../models/fine_tuned_model')
print("Fine-tuned model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned model loaded!


In [2]:
# Load eval dataset
with open('../data/eval_dataset.json', 'r') as f:
    eval_pairs = json.load(f)

jobs_text = [p['job'] for p in eval_pairs]
resumes_text = [p['resume'] for p in eval_pairs]
labels = [p['label'] for p in eval_pairs]

# Generate Embeddings
print("Generating embeddings (Fine-Tuned model)...")
job_embs = model.encode(jobs_text, show_progress_bar=True)
resume_embs = model.encode(resumes_text, show_progress_bar=True)

Generating embeddings (Fine-Tuned model)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [3]:
# Calculate similarities
similarities = []
for i in range(len(eval_pairs)):
    sim = cosine_similarity([job_embs[i]], [resume_embs[i]])[0][0]
    similarities.append(float(sim))

num_jobs = 10
num_resumes = 50

sim_matrix = np.array(similarities).reshape(num_jobs, num_resumes)
label_matrix = np.array(labels).reshape(num_jobs, num_resumes)

precisions = []
recalls = []
f1s = []
mrr_list = []
top5_hits = 0
threshold = 0.45

for i in range(num_jobs):
    sims = sim_matrix[i]
    lbls = label_matrix[i]
    
    sorted_indices = np.argsort(sims)[::-1]
    sorted_lbls = lbls[sorted_indices]
    
    preds = (sims >= threshold).astype(int)
    tp = np.sum((preds == 1) & (lbls == 1))
    fp = np.sum((preds == 1) & (lbls == 0))
    fn = np.sum((preds == 0) & (lbls == 1))
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    
    if np.sum(sorted_lbls[:5]) > 0:
        top5_hits += 1
        
    ranks = np.where(sorted_lbls == 1)[0]
    if len(ranks) > 0:
        mrr_list.append(1.0 / (ranks[0] + 1))
    else:
        mrr_list.append(0.0)

metrics = {
    "Precision": float(np.mean(precisions)),
    "Recall": float(np.mean(recalls)),
    "F1 Score": float(np.mean(f1s)),
    "Top-K Accuracy": float(top5_hits / num_jobs),
    "Mean Similarity Score": float(np.mean(sim_matrix[label_matrix == 1])),
    "Ranking Accuracy (MRR)": float(np.mean(mrr_list))
}

print("Fine-Tuned Model Metrics:")
print(json.dumps(metrics, indent=2))

with open('../data/fine_tuned_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

Fine-Tuned Model Metrics:
{
  "Precision": 0.6232492997198881,
  "Recall": 1.0,
  "F1 Score": 0.7414102564102564,
  "Top-K Accuracy": 1.0,
  "Mean Similarity Score": 0.874830969234011,
  "Ranking Accuracy (MRR)": 1.0
}
